# 머신러닝 과제: 데이터 특성에 따른 분류 모델 선택 및 적용

## 목적
- 제공된 **3개의 데이터셋**을 각각 분석하고, 각 데이터셋에 대해 아래 **3개 모델 중 1개**를 선택하여 학습 및 평가한다.
  - Logistic Regression
  - Decision Tree
  - Random Forest

## 과제 핵심
- 데이터 특성에 기반하여 **왜 그 모델을 선택했는지**, 그리고 **나머지 두 모델을 선택하지 않은 이유**를 논리적으로 설명하는 것이 중요하다.


In [97]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

RANDOM_STATE = 42

---
# 데이터셋 1 (banknote_authentication)


## 1) 데이터셋 설명
- 데이터 출처/의미: 지폐 이미지에서 추출한 웨이브렛(Wavelet) 변환 특성을 바탕으로 진짜(0)와 가짜(1) 지폐를 분류하는 데이터
- 샘플 수: 1372 / 피처 수: 4 (variance, skewness, curtosis, entropy)
- 타깃 변수: class (0: authentic, 1: fake)
- 데이터 타입 특징: 모든 독립 변수가 연속적인 **수치형(Numeric)** 데이터로 구성됨.
- 예상되는 어려움: 피처 간의 통계적 분포 차이가 존재할 수 있으나, 수치형 데이터이므로 모델 선택이 비교적 명확함.

In [ ]:
df1 = pd.read_csv('banknote_authentication.csv')
print('shape:', df1.shape)
display(df1.head())
print('Target distribution:\n', df1['class'].value_counts())
print('Missing values:', df1.isna().sum().sum())

shape: (1372, 5)


,variance,skewness,curtosis,entropy,class
0,3.62160,8.6661,-2.8073,-0.44699,0
1,4.54590,8.1674,-2.4586,-1.46210,0
2,3.86600,-2.6383,1.9242,0.10645,0
3,3.45660,9.5228,-4.0112,-3.59440,0
4,0.32924,-4.4552,4.5718,-0.98880,0


Target distribution:
 class
0    762
1    610
Name: count, dtype: int64
Missing values: 0


## 2) 모델 선택 설명
- **선택한 모델: Logistic Regression**
- **선택 이유**:
  1. 데이터가 모두 **연속적인 수치형** 피처로 구성되어 있어 선형 모델이 학습하기에 매우 적합함.
  2. 복잡한 비선형 관계보다는 각 피처의 가중치 합을 통해 클래스를 구분하는 방식이 효율적임.
  3. 데이터 크기가 크지 않고 피처 수가 적어 오컴의 면도날(Occam's razor) 원칙에 따라 가장 심플한 모델을 우선적으로 선택함.
- **선택하지 않은 모델**:
  - Decision Tree: 수치형 데이터에서 경계를 수직/수평으로만 나누므로 선형 관계 표현에 비효율적일 수 있음.
  - Random Forest: 데이터가 단순하여 강력한 앙상블 모델을 사용하지 않아도 충분한 성능이 확보될 것으로 보임.

In [99]:
X1 = df1.drop('class', axis=1)
y1 = df1['class']
X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=RANDOM_STATE)

model_1 = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=RANDOM_STATE))
])

model_1.fit(X1_train, y1_train)
y1_pred = model_1.predict(X1_test)

print('Accuracy:', accuracy_score(y1_test, y1_pred))
print('\nClassification Report:\n', classification_report(y1_test, y1_pred))

Accuracy: 0.9781818181818182

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.97      0.98       148
           1       0.97      0.98      0.98       127

    accuracy                           0.98       275
   macro avg       0.98      0.98      0.98       275
weighted avg       0.98      0.98      0.98       275



---
# 데이터셋 2 (car_evaluation)


## 1) 데이터셋 설명
- 데이터 출처/의미: 자동차의 구입가, 유지비, 문 개수, 안전도 등을 바탕으로 차량의 수용 가능 여부를 분류
- 샘플 수: 1728 / 피처 수: 6 (buying, maint, doors, persons, lug_boot, safety)
- 타깃 변수: class (unacc, acc, good, vgood)
- 데이터 타입 특징: 모든 입력 피처가 **범주형(Categorical)** 데이터임.
- 예상되는 어려움: 수치적 의미가 없는 텍스트 기반 범주형 데이터이므로 인코딩 전략이 중요함.

In [100]:
df2 = pd.read_csv('car_evaluation.csv')
print('shape:', df2.shape)
display(df2.head())
print('Target distribution:\n', df2['class'].value_counts())
print('Missing values:', df2.isna().sum().sum())

shape: (1728, 7)


,buying,maint,doors,persons,lug_boot,safety,class
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc


Target distribution:
 class
unacc    1210
acc       384
good       69
vgood      65
Name: count, dtype: int64
Missing values: 0


## 2) 모델 선택 설명
- **선택한 모델: Decision Tree**
- **선택 이유**:
  1. 자동차 평가는 "안전도가 낮으면(low) 다른 조건에 상관없이 unacc"와 같은 **계층적 규칙(Hierarchical rules)** 이 명확한 데이터임.
  2. 모든 변수가 범주형이므로, 트리의 각 노드에서 특정 범주에 따라 가지를 치는 방식이 데이터의 생성 원리와 매우 흡사함.
  3. 로지스틱 회귀처럼 원-핫 인코딩 후 고차원으로 변환할 필요 없이 직관적인 분석이 가능함.
- **선택하지 않은 모델**:
  - Logistic Regression: 범주형 변수를 수치화하는 과정에서 선형적 가정(Linear assumption)을 억지로 적용해야 함.
  - Random Forest: 이 데이터는 단일 트리만으로도 충분히 논리적 규칙을 찾아낼 수 있을 만큼 구조가 정형화되어 있음.

In [101]:
X2 = df2.drop('class', axis=1)
y2 = df2['class']

# Target encoding
le = LabelEncoder()
y2_encoded = le.fit_transform(y2)

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2_encoded, test_size=0.2, random_state=RANDOM_STATE)

model_2 = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore')),
    ('classifier', DecisionTreeClassifier(random_state=RANDOM_STATE))
])

model_2.fit(X2_train, y2_train)
y2_pred = model_2.predict(X2_test)

print('Accuracy:', accuracy_score(y2_test, y2_pred))
print('\nClassification Report (Labels:', le.classes_, '):\n', classification_report(y2_test, y2_pred))

Accuracy: 0.9566473988439307

Classification Report (Labels: ['acc' 'good' 'unacc' 'vgood'] ):
               precision    recall  f1-score   support

           0       0.99      0.87      0.92        83
           1       0.62      0.91      0.74        11
           2       0.98      1.00      0.99       235
           3       0.82      0.82      0.82        17

    accuracy                           0.96       346
   macro avg       0.85      0.90      0.87       346
weighted avg       0.96      0.96      0.96       346



---
# 데이터셋 3 (heart_disease)


## 1) 데이터셋 설명
- 데이터 출처/의미: 혈압, 콜레스테롤, 최대 심박수 등 환자의 신체 상태를 바탕으로 심장질환 유무를 분류
- 샘플 수: 303 / 피처 수: 13 (age, sex, cp, thalach 등)
- 타깃 변수: num (0: 정상, 1~4: 질환 단계 -> 0과 1로 이진 분류화하여 사용)
- 데이터 타입 특징: 수치형 변수와 범주형 변수가 혼재되어 있는 **복합 데이터(Mixed types)** 임.
- 예상되는 어려움: 결측치가 존재하며, 다양한 변수 간의 비선형적인 상호작용이 성능에 큰 영향을 미칠 수 있음.

In [102]:
df3 = pd.read_csv('heart_disease.csv')
df3['target'] = df3['num'].apply(lambda x: 1 if x > 0 else 0)
df3 = df3.drop('num', axis=1)

print('shape:', df3.shape)
display(df3.head())
print('Target distribution (0: Normal, 1: Disease):\n', df3['target'].value_counts())
print('Missing values:', df3.isna().sum().sum())

shape: (303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,1
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0


Target distribution (0: Normal, 1: Disease):
 target
0    164
1    139
Name: count, dtype: int64
Missing values: 6


## 2) 모델 선택 설명
- **선택한 모델: Random Forest**
- **선택 이유**:
  1. 의료 데이터 특성상 피처 간의 **복잡한 비선형 상호작용**이 중요하며, 여러 개의 나무를 사용하는 앙상블 기법이 과적합을 방지하면서도 높은 일반화 성능을 제공함.
  2. 수치형(나이, 혈압)과 범주형(성별, 흉통) 변수가 혼합된 환경에서 트리 기반 모델은 스케일 조정에 덜 민감하고 강건한 성능을 보임.
  3. 데이터 양이 적은 편이지만, 부트스트랩 샘플링을 통한 배깅(Bagging) 효과로 모델의 분산(Variance)을 낮출 수 있음.
- **선택하지 않은 모델**:
  - Logistic Regression: 변수 간의 복잡한 비선형 관계를 포착하기 위해 추가적인 다항식 전처리가 필요할 수 있음.
  - Decision Tree: 단일 트리는 의료 데이터의 복잡성으로 인해 훈련 데이터에 과적합(Overfitting)될 위험이 매우 높음.

In [103]:
X3 = df3.drop('target', axis=1)
y3 = df3['target']
X3_train, X3_test, y3_train, y3_test = train_test_split(X3, y3, test_size=0.2, random_state=RANDOM_STATE)

numeric_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
categorical_features = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

model_3 = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE))
])

model_3.fit(X3_train, y3_train)
y3_pred = model_3.predict(X3_test)

print('Accuracy:', accuracy_score(y3_test, y3_pred))
print('\nClassification Report:\n', classification_report(y3_test, y3_pred))

Accuracy: 0.8688524590163934

Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.90      0.87        29
           1       0.90      0.84      0.87        32

    accuracy                           0.87        61
   macro avg       0.87      0.87      0.87        61
weighted avg       0.87      0.87      0.87        61

